# TILTROTOR OpenVSP API
OpenVSP 3.46.0/3.46.02, Python 3.11.15

# API SETUP

# 1 setup CONDA env:

# 1A MacOS setup (supponendo di avere conda già installato): 
con cd ed ls trovare la cartella di OpenVSP fino ad entrare in OpenVSP/python <br>
Entrati nella cartella eseguire i seeguenti comandi (uno alla volta): <br>

conda env create -f ./environment.yml <br>
conda activate vsppytools <br>
pip install -r requirements.txt <br>
conda install -n vsppytools ipykernel -y <br>
python -m ipykernel install --user --name vsppytools --display-name "OpenVSP" <br>

# 1B Windows setup (supponendo di avere conda già installato):
Copiare il percorso completo di OpenVSP/python del vostro pc <br>
Aprirere Anaconda Prompt ed incollare il percorso <br>
Entrati nella cartella eseguire i seeguenti comandi (uno alla volta): <br>

conda env create -f ./environment.yml <br>
conda activate vsppytools <br>
pip install -r requirements.txt <br>
conda install -n vsppytools ipykernel -y <br>
python -m ipykernel install --user --name vsppytools --display-name "OpenVSP" <br>

Se all'ultimo comando per windows si riceve un errore, aprire Anaconda Prompt come amministratore rieseguire tutti i comandi da "conda activate vsppytools" in poi. <br>

# 2 setup kernel con Jupyter Notebook:
command + alt + p con VS Code aperto, cercare 'Developer: Reload Window' e cliccare. I kernel sono aggiornati e cliccando su Kernel in alto a dx vi dovrebbe comparire <br>
"OpenVSP" come opzione.  <br>

# PRE IMPORT 
controllare che OpenVSP non sia in quarantena (soprattutto per nuove installazioni) <br>
xattr .../OpenVSP-3.46.0-MacOS/vsp (in output non ci deve essere nessun .quarantine) <br>
se c'è un .quarantine allora <br>

xattr -rd com.apple.quarantine ~/Desktop/OpenVSP-3.46.0-MacOS <br>
Il tutto dentro conda

# 3 Import Modello ed Input richiesti
Import pacchetti, x estensione della visualizzazione dell'output in VSCode seguire primi commenti della cella successiva ed importare IPython.display, se non si vuole <br>
estendere la visualizzazione dell'output si puo commentare la riga 'from IPython.display import display, HTML' e 'display(...)'. <br>
<br>
I nomi delle geometrie devono combaciare con quelli indicati nella cella seguente. <br>
<br>
Se si sceglie di simulare i propeller con le impostazioni 'disk' o 'both' di OpenVSP bisogna importare le curve di coppia e spinta vs rpm. <br>
<br>
I 'cases' vengono eseguiti in run separate per poter eliminare dal database finale analisi non arrivate a convergenza. <br>
<br>
I singoli 'cases' vengono individuati dal prodotto cartesiano (diadico) dei vettori delle variabili 'alpha', 'beta', 'V_inf', 'Theta_tilt_forw_right', 'Theta_tilt_forw_left', 'RPM_forw_right', 'RPM_forw_left', 'RPM_rear_right', 'RPM_rear_left', 'delta_e', 'delta_r', 'delta_a', avendo ipotizzato unicamente propeller anteriori tiltanti. Mach e Reynolds vengono computati in funzione della V_inf e non sono stati scelti come parametri perchè lo scaling del costo computazionale è esponenziale con l'aumento del numero di dimensioni date dal prodotto diadico.
<br>
Le impostazioni di discretizzazione dei propeller si trovano nella cella 'IMPOSTAZIONE BLADES MODE/DISK MODE/BOTH MODE'. <br>
Per questo codice viene implementata l'opzione 'disk' per le simulazioni di hovering, impostazione 'both' per i casi in transizione e in crociera. <br>
Il numero di Wake Iter e Wake Nodes viene impostato sulla base di valutazioni empiriche sulla convergenza e sugli errori osservati nelle simulazioni. Può essere valutato un futuro rilassamento del costo computazionale. <br>
<br>
L'utente deve inoltre fornire in input la posizione esatta dei propeller nel .vsp3 rispetto all'asse di coordinate del velivolo nella sezione 'DISACCOPPIAMENTO SPINTA DA RISULTANTE AERODINAMICA'.

In [ ]:

import numpy as np
import itertools

from IPython.display import display, HTML

import glob

# Rimuove il limite di output di Jupyter/VS Code
display(HTML("""
<style>
.jp-OutputArea-output { max-height: none !important; }
.output_scroll { height: auto !important; }
</style>
"""))
# Nelle impostazioni di VS Code inoltre apri settings.json (Cmd+Shift+P → "Open User Settings JSON") e aggiungi:
# "notebook.output.textLineLimit": 100000,
# "notebook.output.scrolling": false

#  PERCORSI

MODEL_PATH = "/Users/primianodaddetta/Desktop/modello_vsp/provaprop.vsp3"
OUTPUT_DIR = "/Users/primianodaddetta/Desktop/modello_vsp"

#  NOMI GEOMETRIE

BODY_NAME     = "Vehicle"
PROP_FL_NAME  = "PropAntSx"    # Front-Left  — tiltante
PROP_FR_NAME  = "PropAntDx"    # Front-Right — tiltante
PROP_RL_NAME  = "PropPostSx"   # Rear-Left   — fisso
PROP_RR_NAME  = "PropPostDx"   # Rear-Right  — fisso
HINGE_FL_NAME = "HingeAntSx"
HINGE_FR_NAME = "HingeAntDx"

#  CONDIZIONI ATMOSFERICHE (ISA SL)

RHO     = 1.225    # [kg/m^3]
MU      = 1.789e-5 # [Pa·s]
A_SOUND = 340.3    # [m/s]
SREF    = 0.98     # [m^2]  — da aggiornare eventualmente
CREF    = 0.35     # [m]   — corda di riferimento
BREF    = 2.80     # [m]   — apertura alare

#  CURVA DI ACCOPPIAMENTO ELICA-MOTORE
#  Inserire i valori reali del tuo propeller/motore.
#  rpm_vec : vettore RPM di misura/calcolo
#  ct_vec  : CT corrispondenti
#  cp_vec  : CP corrispondenti
#  D       : diametro propeller [m]

D = 0.3556  # 14" = 0.3556 m

PROP_MAPS_CFG = {
    "FL": dict(rpm_vec=np.array([1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000]),
               ct_vec=np.array([0.0930, 0.0933, 0.0936, 0.0939, 0.0942, 0.0946, 0.0950, 0.0955, 0.0961, 0.0967, 0.0975]),
               cp_vec=np.array([0.0389, 0.0365, 0.0353, 0.0345, 0.0339, 0.0336, 0.0335, 0.334, 0.334, 0.336, 0.0338]),  
               D=D, poly_deg=1),
    "FR": dict(rpm_vec=np.array([1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000]),
               ct_vec=np.array([0.0930, 0.0933, 0.0936, 0.0939, 0.0942, 0.0946, 0.0950, 0.0955, 0.0961, 0.0967, 0.0975]),
               cp_vec=np.array([0.0389, 0.0365, 0.0353, 0.0345, 0.0339, 0.0336, 0.0335, 0.334, 0.334, 0.336, 0.0338]),  
               D=D, poly_deg=1),
    "RL": dict(rpm_vec=np.array([1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000]),
               ct_vec=np.array([0.0930, 0.0933, 0.0936, 0.0939, 0.0942, 0.0946, 0.0950, 0.0955, 0.0961, 0.0967, 0.0975]),
               cp_vec=np.array([0.0389, 0.0365, 0.0353, 0.0345, 0.0339, 0.0336, 0.0335, 0.334, 0.334, 0.336, 0.0338]),  
               D=D, poly_deg=1),
    "RR": dict(rpm_vec=np.array([1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000]),
               ct_vec=np.array([0.0930, 0.0933, 0.0936, 0.0939, 0.0942, 0.0946, 0.0950, 0.0955, 0.0961, 0.0967, 0.0975]),
               cp_vec=np.array([0.0389, 0.0365, 0.0353, 0.0345, 0.0339, 0.0336, 0.0335, 0.334, 0.334, 0.336, 0.0338]),  
               D=D, poly_deg=1),
}

class PropellerMap: # Mapping della curva elica-motore, fitting
    """
    Fitting polinomiale CT=f(RPM), CP=f(RPM) dalla curva elica-motore.
    from_rpm(rpm) restituisce CT, CP, T[N], Q[N·m], eta.
    RPM=0 → propeller spento, tutto zero.
    """
    def __init__(self, cfg):
        rpm, ct, cp = cfg["rpm_vec"], cfg["ct_vec"], cfg["cp_vec"]
        deg = cfg.get("poly_deg", 2)
        self.D         = cfg["D"]
        self._ct_poly  = np.polyfit(rpm, ct, deg)
        self._cp_poly  = np.polyfit(rpm, cp, deg)
        self.rpm_min   = rpm.min()
        self.rpm_max   = rpm.max()

    def from_rpm(self, rpm, rho=RHO):
        if rpm == 0.0:
            return dict(rpm=0.0, CT=0.0, CP=0.0, T=0.0, Q=0.0, eta=0.0)
        n   = rpm / 60.0
        CT  = max(float(np.polyval(self._ct_poly, rpm)), 0.0)
        CP  = max(float(np.polyval(self._cp_poly, rpm)), 1e-9)
        T   = CT * rho * n**2 * self.D**4
        Q   = CP * rho * n**2 * self.D**5 / (2 * np.pi)
        eta = CT / CP
        return dict(rpm=rpm, CT=CT, CP=CP, T=T, Q=Q, eta=eta)

prop_maps = {k: PropellerMap(v) for k, v in PROP_MAPS_CFG.items()}

# Verifica rapida
print("PropellerMap — verifica a 3000 RPM:")
for k, pm in prop_maps.items():
    r = pm.from_rpm(3000)
    print(f"  {k}: CT={r['CT']:.4f}  CP={r['CP']:.5f}  T={r['T']:.2f}N  Q={r['Q']:.4f}N·m  η={r['eta']:.3f}")

# DEFINIZIONE SPAZIO DI ANALISI: prodotto diadico di tutti i vettori 
# Distinzione per modo di volo per ridurre dimensione spazio di analisi.

#  SWEEP HOVERING
#  tilt fisso 0°, no superfici, HoverRamp attivabile (sconsigliato)

HOVER = {
    "alpha_vec"   : np.array([0.0]),
    "beta_vec"    : np.array([0.0]),
    "vinf_vec"    : np.array([14.0]),
    "tilt_FL_vec" : np.array([90.0]),
    "tilt_FR_vec" : np.array([90.0]),
    "rpm_FL_vec"  : np.array([4000.0]),
    "rpm_FR_vec"  : np.array([4000.0]),
    "rpm_RL_vec"  : np.array([4000.0]),
    "rpm_RR_vec"  : np.array([4000.0]),
    "delta_e_vec" : np.array([0.0]),
    "delta_r_vec" : np.array([0.0]),
    "delta_a_vec" : np.array([0.0]),
}

#  SWEEP TRANSIZIONE
#  tilting attivo, tutti i motori liberi

TRANSITION = {
    "alpha_vec"   : np.array([5.0]),
    "beta_vec"    : np.array([0.0]),
    "vinf_vec"    : np.array([16.0]),
    "tilt_FL_vec" : np.array([30.0]),
    "tilt_FR_vec" : np.array([30.0]),
    "rpm_FL_vec"  : np.array([3000.0]),
    "rpm_FR_vec"  : np.array([3000.0]),
    "rpm_RL_vec"  : np.array([3000.0]),
    "rpm_RR_vec"  : np.array([3000.0]),
    "delta_e_vec" : np.array([5.0]),
    "delta_r_vec" : np.array([0.0]),
    "delta_a_vec" : np.array([0.0]),
}

#  SWEEP CROCIERA
#  tilt anteriori fissi 90°, posteriori RPM=0

CRUISE = {
    "alpha_vec"   : np.array([0.0]),
    "beta_vec"    : np.array([0.0]),
    "vinf_vec"    : np.array([22.0]),
    "tilt_FL_vec" : np.array([0.0]),
    "tilt_FR_vec" : np.array([0.0]),
    "rpm_FL_vec"  : np.array([2000.0]),
    "rpm_FR_vec"  : np.array([2000.0]),
    "rpm_RL_vec"  : np.array([1.0]),   # motori posteriori spenti, NON METTERE 0 PERCHE' OPENVSP NON CONVERGE ALTRIMENTI, SETTATO 1 RPM SIMBOLICO
    "rpm_RR_vec"  : np.array([1.0]),
    "delta_e_vec" : np.array([0.0]),
    "delta_r_vec" : np.array([0.0]),
    "delta_a_vec" : np.array([0.0]),
}

# prodotto cartesiano
def generate_cases(sweep, phase):
    keys = ["alpha","beta","vinf",
            "tilt_FL","tilt_FR",
            "rpm_FL","rpm_FR","rpm_RL","rpm_RR",
            "delta_e","delta_r","delta_a"]
    vecs = [sweep[f"{k}_vec"] for k in keys]
    cases = []
    for combo in itertools.product(*vecs):
        d = dict(zip(keys, (float(v) for v in combo)))
        d["phase"] = phase
        # Re e Mach da Vinf
        d["Re"]   = RHO * d["vinf"] * CREF / MU
        d["Mach"] = d["vinf"] / A_SOUND
        # CT/CP/T/Q da PropellerMap
        for key in ["FL","FR","RL","RR"]:
            r = prop_maps[key].from_rpm(d[f"rpm_{key}"])
            d[f"ct_{key}"] = r["CT"]
            d[f"cp_{key}"] = r["CP"]
            d[f"T_{key}"]  = r["T"]
            d[f"Q_{key}"]  = r["Q"]
        # Nome univoco 
        d["name"] = (
            f"{phase}"
            f"_a{d['alpha']:+.0f}"
            f"_b{d['beta']:+.0f}"
            f"_V{d['vinf']:.0f}"
            f"_tFL{d['tilt_FL']:.0f}"
            f"_tFR{d['tilt_FR']:.0f}"
            f"_wFL{d['rpm_FL']:.0f}"
            f"_wFR{d['rpm_FR']:.0f}"
            f"_wRL{d['rpm_RL']:.0f}"
            f"_wRR{d['rpm_RR']:.0f}"
            f"_de{d['delta_e']:+.0f}"
            f"_dr{d['delta_r']:+.0f}"
            f"_da{d['delta_a']:+.0f}"
        ).replace("+","p").replace("-","m")
        cases.append(d)
    return cases

hover_cases      = generate_cases(HOVER,      "hover")
transition_cases = generate_cases(TRANSITION, "transition")
cruise_cases     = generate_cases(CRUISE,     "cruise")
ALL_CASES        = hover_cases + transition_cases + cruise_cases

print(f"\nCasi generati:")
print(f"  Hovering    : {len(hover_cases)}")
print(f"  Transizione : {len(transition_cases)}")
print(f"  Crociera    : {len(cruise_cases)}")
print(f"  TOTALE      : {len(ALL_CASES)}")


PropellerMap — verifica a 3000 RPM:
  FL: CT=0.0936  CP=0.04399  T=4.58N  Q=0.1219N·m  η=2.127
  FR: CT=0.0936  CP=0.04399  T=4.58N  Q=0.1219N·m  η=2.127
  RL: CT=0.0936  CP=0.04399  T=4.58N  Q=0.1219N·m  η=2.127
  RR: CT=0.0936  CP=0.04399  T=4.58N  Q=0.1219N·m  η=2.127

Casi generati:
  Hovering    : 1
  Transizione : 1
  Crociera    : 1
  TOTALE      : 3


# IMPORT VERIFICA AMBIENTE

In [2]:

import openvsp as vsp

print(vsp.__file__)
print(dir(vsp))

import os, shutil, sys
import numpy as np

print(f"Python  : {sys.version.split()[0]}")
print(f"OpenVSP : {vsp.GetVSPVersion()}")
print(f"CWD     : {os.getcwd()}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

/Users/primianodaddetta/anaconda3/envs/vsppytools/lib/python3.11/site-packages/openvsp/__init__.py
['ABS', 'ALIGN_BOTTOM', 'ALIGN_CENTER', 'ALIGN_LEFT', 'ALIGN_MIDDLE', 'ALIGN_PIXEL', 'ALIGN_RIGHT', 'ALIGN_TOP', 'ALL_DIR', 'ALL_GDEV_TYPES', 'ALL_GEOM_SCREENS', 'ANG_0', 'ANG_180', 'ANG_270', 'ANG_90', 'ANG_DEG', 'ANG_RAD', 'APPROX_CEDIT', 'AREA_WSECT_DRIVER', 'AREA_XSEC_DRIVER', 'AR_WSECT_DRIVER', 'ATMOS_TYPE_HERRINGTON_1966', 'ATMOS_TYPE_MANUAL_P_R', 'ATMOS_TYPE_MANUAL_P_T', 'ATMOS_TYPE_MANUAL_RE_L', 'ATMOS_TYPE_MANUAL_R_T', 'ATMOS_TYPE_US_STANDARD_1976', 'ATTACH_ROT_COMP', 'ATTACH_ROT_EtaMN', 'ATTACH_ROT_LMN', 'ATTACH_ROT_NONE', 'ATTACH_ROT_NUM_TYPES', 'ATTACH_ROT_RST', 'ATTACH_ROT_UV', 'ATTACH_TRANS_COMP', 'ATTACH_TRANS_EtaMN', 'ATTACH_TRANS_LMN', 'ATTACH_TRANS_NONE', 'ATTACH_TRANS_NUM_TYPES', 'ATTACH_TRANS_RST', 'ATTACH_TRANS_UV', 'ATTROBJ_ADVLINK', 'ATTROBJ_ATTR', 'ATTROBJ_COLLECTION', 'ATTROBJ_FREE', 'ATTROBJ_GEOM', 'ATTROBJ_LINK', 'ATTROBJ_MEASURE', 'ATTROBJ_MODE', 'ATTROBJ_PARM'

# LOADING E GEOM MAPPING
Parti caricate all'interno del modello e visualizzate correttamente dal codice

In [3]:

vsp.VSPCheckSetup()
vsp.VSPRenew()
vsp.ReadVSPFile(MODEL_PATH)
vsp.Update()

# print geom nel modello
print(f"\n{'Nome':<25} {'Tipo':<20} {'ID'}")
print("-" * 75)
for gid in vsp.FindGeoms():
    print(f"{vsp.GetGeomName(gid):<25} {vsp.GetGeomTypeName(gid):<20} {gid}")


Nome                      Tipo                 ID
---------------------------------------------------------------------------
WingGeom                  Wing                 EIHJXOMLTA
Horizontal Tail           Wing                 OOQXQONAEL
Batteria_1                Custom               AHYOSTGSRG
Batteria_2                Custom               QPRBAWFDGQ
Acqua                     Custom               IXLUBLUGJF
FuselageGeom              Fuselage             ITOOIUUFWH
Paracadute                Custom               DNKAPBVDHZ
Vertical Tail             Wing                 XCSXPKAJLN
Motore ant dx             Custom               GYKYEFQQHZ
Motore ant sx             Custom               AVVHBJLUAP
Motore post sx            Custom               AKRZEIEFEN
Motore post dx            Custom               FQFHCFPOLL
Pompa                     Custom               IVGYZBBVXE
Elettrovalvola            Custom               SOJNBOOBUA
Camera                    Custom               OPDTPINYRX
Vid

# GERARCHIA HINGE-PROPELLER
Propeller va inserito nel modello OpenVSP come child della hinge vedi: https://www.nasa.gov/reference/openvsp-hinges/ <br>
Questo pezzo di codice verifica parent-child di propeller e hinge (ipotizzato tiltrotor semplice, no foldable wing, no qualsiasi altra parte mobile)

In [4]:

print("\nVerifica parent-child:")
for pname in [PROP_FL_NAME, PROP_FR_NAME, PROP_RL_NAME, PROP_RR_NAME]:
    geom_ids = vsp.FindGeomsWithName(pname)
    if not geom_ids:
        print("NON TROVATO nel modello!")
        continue
    gid = geom_ids[0]
    parent_id = vsp.GetGeomParent(gid)
    if parent_id == "" or parent_id is None or parent_id == "NONE":
        parent_name = "NESSUNO (root)"
    else:
        parent_name = vsp.GetGeomName(parent_id)
    print(f"  {pname:<20} → parent: {parent_name}")

print("\nVerifica Hinge:")
for hname in [HINGE_FL_NAME, HINGE_FR_NAME]:
    geom_ids = vsp.FindGeomsWithName(hname)
    if not geom_ids:
        print("NON TROVATO nel modello!")
        continue
    hid = geom_ids[0]
    pid = vsp.FindParm(hid, "JointRotate", "Hinge")
    if pid == "":
        print(f"  {hname:<20} → nessun parm JointRotate/Hinge trovato")
        continue
    angle = vsp.GetParmVal(pid)
    print(f"  {hname:<20} → angolo attuale: {angle:.2f}°")



Verifica parent-child:
  PropAntSx            → parent: HingeAntSx
  PropAntDx            → parent: HingeAntDx
  PropPostSx           → parent: NESSUNO (root)
  PropPostDx           → parent: NESSUNO (root)

Verifica Hinge:
  HingeAntSx           → angolo attuale: 60.00°
  HingeAntDx           → angolo attuale: 0.00°


# UTILITIES
Miscellanea 

In [ ]:

def set_tilt(hinge_name, angle_deg):
    hid = vsp.FindGeomsWithName(hinge_name)[0]
    vsp.SetParmVal(vsp.FindParm(hid, "JointRotate", "Hinge"), angle_deg)

def set_rotor_params(disk_index, rpm, ct, cp):
    """Imposta RPM, CT, CP su un actuator disk specifico."""
    did = vsp.FindActuatorDisk(disk_index)
    vsp.SetParmVal(vsp.FindParm(did, "RotorRPM", "Rotor"), rpm)
    vsp.SetParmVal(vsp.FindParm(did, "RotorCT",  "Rotor"), ct)
    vsp.SetParmVal(vsp.FindParm(did, "RotorCP",  "Rotor"), cp)

def set_prop_mode(prop_name, mode):
    pid = vsp.FindGeomsWithName(prop_name)[0]
    vsp.SetParmVal(vsp.FindParm(pid, "PropMode", "Design"), mode)

def verify_disk_normals():
    n = vsp.GetNumActuatorDisks()
    print(f"\n  Dischi trovati: {n}")

#  Re e Mach da Vinf
def compute_re_mach(vinf):
    return RHO * vinf * CREF / MU, vinf / A_SOUND

# parse_stab_file 
def parse_stab_file(stab_path):
    """
    Parsa TUTTI i blocchi alpha nel file .stab (separati da ***...).
    
    Restituisce una lista di dict — un dict per punto alpha — con:
      - scalari: AoA, Beta_cond, Vinf_cond, Mach_cond, Rho, SM, X_np
      - coefficienti base: CL, CD, CS, CFx, CFy, CFz, CMl, CMm, CMn
      - derivate: CL_Alpha, CL_Beta, CL_p, CL_q, CL_r, CL_Mach,
                  CL_ConGrp1, CL_ConGrp2, CL_ConGrp3  (e analoghi per ogni coeff)
    
    I nomi ConGrp_1/2/3 nel .stab corrispondono a Ailerons/Elevator/Rudder
    nell'ordine in cui sono definiti nel modello OpenVSP.
    """
    import re
    if not os.path.exists(stab_path):
        print(f"  [WARN] .stab non trovato: {stab_path}")
        return []

    with open(stab_path, "r") as f:
        raw = f.read()

    blocks = re.split(r"\*{10,}", raw)
    blocks = [b.strip() for b in blocks if b.strip()]

    SCALARS = {
        "Sref_":"Sref", "Cref_":"Cref", "Bref_":"Bref",
        "Xcg_":"Xcg",   "Ycg_":"Ycg",   "Zcg_":"Zcg",
        "Mach_":"Mach_cond", "AoA_":"AoA", "Beta_":"Beta_cond",
        "Rho_":"Rho",   "Vinf_":"Vinf_cond",
        "SM":"SM",      "X_np":"X_np",
    }

    results = []
    for block in blocks:
        rec  = {}
        lines = block.splitlines()

        # scalari header
        for line in lines:
            if line.strip().startswith("#") or not line.strip():
                continue
            tok = line.split()
            if len(tok) >= 2 and tok[0] in SCALARS:
                try:
                    rec[SCALARS[tok[0]]] = float(tok[1])
                except ValueError:
                    pass

        # tabella derivate
        hdr_idx, col_names = None, []
        for i, line in enumerate(lines):
            if "Coef" in line and "Alpha" in line and "Beta" in line:
                hdr_idx   = i
                col_names = line.split()
                break

        if hdr_idx is not None:
            suffix_map = {
                "Alpha":   "_Alpha",
                "Beta":    "_Beta",
                "p":       "_p",
                "q":       "_q",
                "r":       "_r",
                "Mach":    "_Mach",
                "U":       "_U",
                "ConGrp1": "_ConGrp1",
                "ConGrp2": "_ConGrp2",
                "ConGrp3": "_ConGrp3",
            }
            for line in lines[hdr_idx + 1:]:
                stripped = line.strip()
                if not stripped or stripped.startswith("#"):
                    continue
                if stripped.startswith("TITLE"):
                    continue
                tok = stripped.split()
                if len(tok) < 3:
                    break
                coef = tok[0]
                try:
                    rec[coef] = float(tok[1])
                except ValueError:
                    continue
                for j, col in enumerate(col_names[2:], start=2):
                    if j >= len(tok):
                        break
                    suf = suffix_map.get(col)
                    if suf is None:
                        continue
                    try:
                        rec[f"{coef}{suf}"] = float(tok[j])
                    except ValueError:
                        pass

        rec.setdefault("SM",   float("nan"))
        rec.setdefault("X_np", float("nan"))
        
        if rec:
            results.append(rec)

    return results   # lista di dict, uno per punto alpha


# check_convergence dal file .history 
import math
MIN_ITER = 3
CONV_KEYS   = ["CLtot", "CMytot"]

def check_convergence(history_path, min_iter=MIN_ITER):
    if not os.path.exists(history_path):
        return dict(converged=False, reason="history non trovato",
                    last_vals={}, rel_var={}, n_iter=0)

    rows, header = [], None
    with open(history_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            tok = line.split()
            if header is None:
                header = tok
                continue
            if len(tok) < len(header):
                continue
            try:
                row = {header[i]: float(tok[i]) for i in range(len(header))}
                rows.append(row)
            except (ValueError, IndexError):
                continue

    n_iter = len(rows)
    if n_iter < min_iter:
        return dict(converged=False,
                    reason=f"iterazioni insufficienti ({n_iter} < {min_iter})",
                    last_vals={}, rel_var={}, n_iter=n_iter)

    has_nan = any(
        math.isnan(v) or math.isinf(v)
        for row in rows for v in row.values()
    )

    last = rows[-1]
    last_vals = {k: last.get(k, float("nan")) for k in CONV_KEYS}

    return dict(
        converged=not has_nan,
        reason="OK" if not has_nan else "NaN/Inf nel history",
        last_vals=last_vals,
        rel_var={},
        n_iter=n_iter
    )


#  convergenza (PNG per ogni caso)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def plot_convergence(history_path, case_name, out_dir):
    if not os.path.exists(history_path):
        return
    data, header = {}, None
    with open(history_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            tok = line.split()
            if header is None:
                header = tok
                for k in header:
                    data[k] = []
                continue
            if len(tok) < len(header):
                continue
            try:
                for i, k in enumerate(header):
                    data[k].append(float(tok[i]))
            except ValueError:
                continue

    plot_keys = [k for k in CONV_KEYS if k in data and len(data[k]) > 0]
    if not plot_keys:
        return
    iters = np.arange(1, len(data[plot_keys[0]]) + 1)
    fig, ax = plt.subplots(figsize=(7, 3))
    for k in plot_keys:
        ax.semilogy(iters, np.abs(data[k]) + 1e-16, label=k)
    ax.set_xlabel("Iterazione")
    ax.set_ylabel("|Coeff| (log)")
    ax.set_title(f"Convergenza — {case_name}")
    ax.legend(); ax.grid(True, which="both", ls="--", alpha=0.5)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"{case_name}_conv.png"), dpi=110)
    plt.close(fig)

# Mappa group name → geom_id + subsurf_id
# Da costruire una volta sola dopo ReadVSPFile
def build_cs_parm_map():
    """
    Mappa nome SubSurf SS_CONTROL → parm_id di StartAngle.
    Con SameAngleFlag=1 (default), impostare StartAngle imposta anche EndAngle.
    """
    cs_map = {}
    for gid in vsp.FindGeoms():
        for ss_id in vsp.GetSubSurfIDVec(gid):
            if vsp.GetSubSurfType(ss_id) != vsp.SS_CONTROL:
                continue
            ss_name = vsp.GetSubSurfName(ss_id)
            pid = vsp.FindParm(ss_id, "StartAngle", "SS_Control")
            if pid:
                cs_map[ss_name] = pid
    print("CS trovate:", list(cs_map.keys()))
    return cs_map

CS_PARM_MAP = build_cs_parm_map()

def clean_degengeom(model_dir):
    patterns = ["*_DegenGeom.csv", "*_DegenGeom.vspaero", "*.vspgeom"]
    for pat in patterns:
        for f in glob.glob(os.path.join(model_dir, pat)):
            os.remove(f)


print("CS trovate:", list(CS_PARM_MAP.keys()))

print("Utilities caricate.")


CS trovate: ['Ailerons', 'Elevator', 'Rudder']
CS trovate: ['Ailerons', 'Elevator', 'Rudder']
Utilities caricate.


# IMPOSTAZIONE BLADES MODE/DISK MODE/BOTH MODE
Richiesto input utente. <br>
<br>
'disk' -> disco attuatore, molto veloce, da quello che ho visto personalmente anche scarsamente convergente, analisi steady, fornisce .stab <br>
'blades' -> BET, possibile scegliere sia steady che unsteady (non in questo codice), chiaramente + lento, unsteady non da .stab, <br>
'both' -> via di mezzo, tra i due casi precedenti, mi ha dato l'impressione di funzionare bene ed infatti è impostato di default.

In [6]:
import ipywidgets as widgets
from IPython.display import display

# Modalità propeller: "disk" | "blades" | "both"
PROPMODE_BY_PHASE = {
    "hover":      "disk",   # PROPDISK: unico che parte in hover
    "transition": "both",   # PROPBOTH: migliore per transizione
    "cruise":     "both",   # PROPBOTH: richiesto per crociera
}

prop_mode_map = {"disk": vsp.PROP_DISK, "blades": vsp.PROP_BLADES, "both": vsp.PROP_BOTH}

mode_labels = {vsp.PROP_DISK: "disk", vsp.PROP_BLADES: "blades", vsp.PROP_BOTH: "both"}

In [7]:

vsp.SetAnalysisInputDefaults("VSPAEROSweep")
print(vsp.GetAnalysisInputNames("VSPAEROSweep"))

('2DFEMFlag', 'AdjointGMRESConvergenceFactor', 'AlphaEnd', 'AlphaNpts', 'AlphaStart', 'AutoTimeNumRevs', 'AutoTimeStepFlag', 'BetaEnd', 'BetaNpts', 'BetaStart', 'CGDegenSet', 'CGGeomSet', 'CGModeID', 'CLMax2D', 'Clo2D', 'CoreSizeFactor', 'FarAway', 'FarDist', 'FarDistToggle', 'FixedWakeFlag', 'ForwardGMRESConvergenceFactor', 'FreezeMultiPoleAtIteration', 'FreezeWakeAtIteration', 'FreezeWakeRootVortices', 'FromSteadyState', 'GeomSet', 'GroundEffect', 'GroundEffectToggle', 'HoverRamp', 'HoverRampFlag', 'ImplicitWake', 'ImplicitWakeStartIteration', 'MACFlag', 'MachEnd', 'MachNpts', 'MachStart', 'Machref', 'ManualVrefFlag', 'MassSliceDir', 'ModeID', 'NCPU', 'NoiseCalcFlag', 'NoiseCalcType', 'NoiseUnits', 'NonLinearConvergenceFactor', 'NumMassSlice', 'NumTimeSteps', 'NumWakeNodes', 'PropBladesMode', 'QuadTreeBufferLevels', 'ReCref', 'ReCrefEnd', 'ReCrefNpts', 'RedirectFile', 'RefFlag', 'Rho', 'ScurveFlag', 'Sref', 'StallModel', 'StartAveragingTimeStep', 'StopBeforeRun', 'Symmetry', 'Tecplot

Creazione di un monitor prima dell'analisi in VSPAero.

In [8]:

import psutil
import threading
import time
import traceback

_monitor_active = True

def monitor_vspaero(interval=5, initial_delay=8):
    """
    Monitora il processo vspaero ogni `interval` secondi.
    initial_delay: attendi prima del primo check (il processo ci mette ~5-8s ad apparire).
    """
    import time, subprocess
    time.sleep(initial_delay)   # ← aspetta che vspaero sia partito
    while True:
        try:
            result = subprocess.run(
                ["pgrep", "-a", "vspaero"],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                print(f"  [monitor] nessun vspaero attivo — {datetime.now().strftime('%H:%M:%S')}")
                break
            # parsing CPU/MEM da ps
            pids = [l.split()[0] for l in result.stdout.strip().splitlines()]
            for pid in pids:
                ps = subprocess.run(
                    ["ps", "-p", pid, "-o", "pid,%cpu,rss="],
                    capture_output=True, text=True
                )
                lines = ps.stdout.strip().splitlines()
                if len(lines) > 1:
                    _, cpu, mem = lines[1].split()
                    mem_mb = int(mem) // 1024
                    print(f"  [monitor] vspaero PID={pid} CPU={cpu}% MEM={mem_mb}MB "
                          f"status=running")
        except Exception:
            break
        time.sleep(interval)

# Avvia il monitor in background
_monitor_thread = threading.Thread(target=monitor_vspaero,
                                   args=(5,), daemon=True)
_monitor_thread.start()
print("Monitor avviato — stampa ogni 5s. Ora esegui il Main Loop.")

inputs = vsp.GetAnalysisInputNames("VSPAEROSweep")
congrp = [x for x in inputs if "con" in x.lower() or "ctrl" in x.lower()
    or "surf" in x.lower() or "delta" in x.lower() or "group" in x.lower()]
print("Input relativi a superfici di controllo:")
for x in congrp:
    print(f"  {x}")
print("\nTutti gli input:")
for x in sorted(inputs):
    print(f"  {x}")

# Mapping nome propeller → indice disco VSPAero
# L'ordine dipende da come OpenVSP assegna gli indici.
# Verificare stampando: [vsp.FindActuatorDisk(i) for i in range(4)]
# e confrontando con i nomi dei propeller nel modello.
DISK_INDEX = {"FL": 0, "FR": 1, "RL": 2, "RR": 3}

Monitor avviato — stampa ogni 5s. Ora esegui il Main Loop.
Input relativi a superfici di controllo:
  AdjointGMRESConvergenceFactor
  ForwardGMRESConvergenceFactor
  NonLinearConvergenceFactor
  UpdateMatrixPreconditioner
  UseWakeNodeMatrixPreconditioner

Tutti gli input:
  2DFEMFlag
  AdjointGMRESConvergenceFactor
  AlphaEnd
  AlphaNpts
  AlphaStart
  AutoTimeNumRevs
  AutoTimeStepFlag
  BetaEnd
  BetaNpts
  BetaStart
  CGDegenSet
  CGGeomSet
  CGModeID
  CLMax2D
  Clo2D
  CoreSizeFactor
  FarAway
  FarDist
  FarDistToggle
  FixedWakeFlag
  ForwardGMRESConvergenceFactor
  FreezeMultiPoleAtIteration
  FreezeWakeAtIteration
  FreezeWakeRootVortices
  FromSteadyState
  GeomSet
  GroundEffect
  GroundEffectToggle
  HoverRamp
  HoverRampFlag
  ImplicitWake
  ImplicitWakeStartIteration
  MACFlag
  MachEnd
  MachNpts
  MachStart
  Machref
  ManualVrefFlag
  MassSliceDir
  ModeID
  NCPU
  NoiseCalcFlag
  NoiseCalcType
  NoiseUnits
  NonLinearConvergenceFactor
  NumMassSlice
  NumTimeSteps
  

# MAIN LOOP
Analisi dell'intero spazio. QUALORA VENGANO APPORTATE MODIFICHE L'ORDINE DEI COMANDI 'SetIntAnalysisInput' e 'SetDoubleAnalysisInput' DEVE RIMANERE INVARIATO. <br>
'SetIntAnalysisInput' e 'SetDoubleAnalysisInput' possiedono un ordine preciso con cui possono essere chiamati, l'errata chiamata di uno di essi causa sovrascrittura di tutte le entrate precedenti.

In [ ]:

import shutil
from datetime import datetime

model_basename = os.path.splitext(os.path.basename(MODEL_PATH))[0]
model_dir      = os.path.dirname(os.path.abspath(MODEL_PATH))

stab_results = {}   # case_name → lista di dict (uno per punto alpha)
results_log  = []   # metadati di ogni caso (convergenza, parametri, ecc.)

START_TIME = datetime.now()
print(f"Inizio analisi: {START_TIME.strftime('%H:%M:%S')}")
print(f"Casi totali   : {len(ALL_CASES)}\n")

for idx, case in enumerate(ALL_CASES):
    cname    = case["name"]
    phase    = case["phase"]
    case_dir = os.path.join(OUTPUT_DIR, phase, cname)
    os.makedirs(case_dir, exist_ok=True)

    print(f"[{idx+1:04d}/{len(ALL_CASES)}] {cname[:80]}")
    print(f"  >>> INIZIO CASO {idx+1}")

    orig_dir = os.getcwd()
    os.chdir(case_dir)

    try:
        print(f"  >>> DENTRO TRY")

        clean_degengeom(model_dir)

        phase         = case["phase"]
        mode_str      = PROPMODE_BY_PHASE.get(phase, "both") # forza 'both' per fasi non definite in 'PROPMODE_BY_PHASE'
        propmode      = prop_mode_map[mode_str]

        wake_cfg = {
            "hover":      {"WakeNumIter": 16, "NumWakeNodes": 64},
            "transition": {"WakeNumIter": 16, "NumWakeNodes": 64},
            "cruise":     {"WakeNumIter": 12, "NumWakeNodes": 64},
        }
        wcfg = wake_cfg.get(phase, {"WakeNumIter": 15, "NumWakeNodes": 32}) # forza 15 WakeIter e 32 WakeNodes per fasi non definite in 'wake_cfg'

        for pname in [PROP_FL_NAME, PROP_FR_NAME, PROP_RL_NAME, PROP_RR_NAME]:
            set_prop_mode(pname, propmode)
        vsp.Update()
        
        # Tilting hinge anteriori 
        print(f"  >>> set_tilt FL")
        set_tilt(HINGE_FL_NAME, case["tilt_FL"])
        print(f"  >>> set_tilt FR")
        set_tilt(HINGE_FR_NAME, case["tilt_FR"])
        vsp.Update()
        print(f"  >>> Update OK")

        # RPM / CT / CP sui 4 dischi 
        print(f"  >>> set_rotor_params")
        for key, didx in DISK_INDEX.items():
            set_rotor_params(didx,
                             case[f"rpm_{key}"],
                             case[f"ct_{key}"],
                             case[f"cp_{key}"])
        print(f"  >>> rotori OK")

        # Deflessioni CS via SetParmVal 
        cs_deflections = {
            "Ailerons" : case["delta_a"],
            "Elevator" : case["delta_e"],
            "Rudder"   : case["delta_r"],
        }
        for cs_name, val in cs_deflections.items():
            pid = CS_PARM_MAP.get(cs_name)
            if pid:
                vsp.SetParmVal(pid, val)
            else:
                print(f"  [WARN] CS '{cs_name}' non trovata nella mappa")
        vsp.Update()
        print(f"  >>> CS OK")

        vsp.SetAnalysisInputDefaults("MassProp")
        vsp.SetIntAnalysisInput("MassProp", "NumMassSlice", [100], 0)
        mass_csv = os.path.join(case_dir, f"{cname}_massprop.csv")
        vsp.SetComputationFileName(vsp.MASS_PROP_TXT_TYPE, mass_csv)
        rid_mass = vsp.ExecAnalysis("MassProp")
        xcg = vsp.GetDoubleResults(rid_mass, "Total_CG_x", 0)[0]
        ycg = vsp.GetDoubleResults(rid_mass, "Total_CG_y", 0)[0]
        zcg = vsp.GetDoubleResults(rid_mass, "Total_CG_z", 0)[0]
        print(f"  MassProp OK → CG=({xcg:.4f}, {ycg:.4f}, {zcg:.4f})")

        print(f"  >>> VSPAEROSweep start")
        # VSPAEROSweep 
        analysis = "VSPAEROSweep"
        vsp.SetAnalysisInputDefaults(analysis)

        N_CPU = psutil.cpu_count(logical = False)
        vsp.SetIntAnalysisInput("VSPAEROSweep", "NCPU", [N_CPU])

        vsp.SetIntAnalysisInput(analysis, "WakeNumIter",  [wcfg["WakeNumIter"]],  0)
        vsp.SetIntAnalysisInput(analysis, "NumWakeNodes", [wcfg["NumWakeNodes"]], 0)

        hover_flag = 0

        vsp.SetIntAnalysisInput(analysis, "Symmetry",      [0],          0)
        vsp.SetIntAnalysisInput(analysis, "HoverRampFlag", [hover_flag], 0)

        vsp.SetDoubleAnalysisInput(analysis, "AlphaStart", [case["alpha"]], 0)
        vsp.SetDoubleAnalysisInput(analysis, "AlphaEnd",   [case["alpha"]], 0)
        vsp.SetIntAnalysisInput(analysis,    "AlphaNpts",  [1],             0)

        vsp.SetDoubleAnalysisInput(analysis, "BetaStart",  [case["beta"]],  0)
        vsp.SetDoubleAnalysisInput(analysis, "BetaEnd",    [case["beta"]],  0)
        vsp.SetIntAnalysisInput(analysis,    "BetaNpts",   [1],             0)

        vsp.SetDoubleAnalysisInput(analysis, "MachStart",  [case["Mach"]],  0)
        vsp.SetDoubleAnalysisInput(analysis, "MachEnd",    [case["Mach"]],  0)
        vsp.SetIntAnalysisInput(analysis,    "MachNpts",   [1],             0)
        vsp.SetDoubleAnalysisInput(analysis, "Vinf",       [case["vinf"]],  0)
        vsp.SetDoubleAnalysisInput(analysis, "Rho",        [RHO],           0)

        vsp.SetDoubleAnalysisInput(analysis, "Xcg", [xcg])  # ← CG da MassProp
        vsp.SetDoubleAnalysisInput(analysis, "Ycg", [ycg])
        vsp.SetDoubleAnalysisInput(analysis, "Zcg", [zcg])

        # analisi 
        rid = vsp.ExecAnalysis(analysis)
        print(f"  >>> DOPO EXECANALYSIS rid={rid}")
        print(f"  ExecAnalysis completato, rid={rid}")

        import shutil

        hist_src  = os.path.join(model_dir, f"{model_basename}.history")
        stab_src  = os.path.join(model_dir, f"{model_basename}.stab")
        hist_dest = os.path.join(case_dir,  f"{cname}.history")
        stab_dest = os.path.join(case_dir,  f"{cname}.stab")

        # Sposta .history
        conv_res = dict(converged=False, reason="history non trovato",
                        last_vals={}, rel_var={}, n_iter=0)
        if os.path.exists(hist_src):
            shutil.move(hist_src, hist_dest)
            print(f"  hist copiato: {hist_dest}")
            conv_res = check_convergence(hist_dest)
            plot_convergence(hist_dest, cname, case_dir)
        else:
            print(f"  [WARN] hist non trovato in model_dir: {hist_src}")

        # Sposta .stab
        stab_data = []
        if os.path.exists(stab_src):
            shutil.move(stab_src, stab_dest)
            print(f"  stab copiato: {stab_dest}")
            stab_data = parse_stab_file(stab_dest)
        else:
            print(f"  [WARN] stab non trovato in model_dir: {stab_src}")

        stab_results[cname] = stab_data

        vsp.SetAnalysisInputDefaults("ParasiteDrag")
        vsp.SetDoubleAnalysisInput("ParasiteDrag", "Vinf", [case["vinf"]])
        vsp.SetDoubleAnalysisInput("ParasiteDrag", "Mach", [case["Mach"]])
        vsp.SetDoubleAnalysisInput("ParasiteDrag", "Rho",  [RHO])
        cd0_csv = os.path.join(case_dir, f"{cname}_cd0.csv")
        vsp.SetComputationFileName(vsp.DRAG_BUILD_CSV_TYPE, cd0_csv)
        vsp.ExecAnalysis("ParasiteDrag")
        print(f"  ParasiteDrag OK → {cd0_csv}")

        # Log metadati caso 
        log_entry = {
            **{k: case[k] for k in [
                "name","phase","alpha","beta","vinf","Mach","Re",
                "tilt_FL","tilt_FR",
                "rpm_FL","rpm_FR","rpm_RL","rpm_RR",
                "ct_FL","ct_FR","ct_RL","ct_RR",
                "cp_FL","cp_FR","cp_RL","cp_RR",
                "T_FL","T_FR","T_RL","T_RR",
                "Q_FL","Q_FR","Q_RL","Q_RR",
                "delta_e","delta_r","delta_a",
            ]},
            "converged":   conv_res["converged"],
            "conv_reason": conv_res["reason"],
            "n_iter":      conv_res["n_iter"],
            "stab_blocks": len(stab_data),
            "case_dir":    case_dir,
        }
        results_log.append(log_entry)

        status = "conv" if conv_res["converged"] else "no-conv"
        print(f"{status} | iter={conv_res['n_iter']} "
              f"| blocks={len(stab_data)} "
              f"| α={case['alpha']:+.1f}° β={case['beta']:+.1f}° "
              f"V={case['vinf']:.0f}m/s")

    except Exception as e:
        print(f"  ERRORE: {e}")
        traceback.print_exc()
        results_log.append({
            **{k: case.get(k, None) for k in ["name","phase","alpha","beta",
               "vinf","tilt_FL","tilt_FR","rpm_FL","rpm_FR","rpm_RL","rpm_RR",
               "delta_e","delta_r","delta_a"]},
            "converged": False, "conv_reason": str(e),
            "n_iter": 0, "stab_blocks": 0, "case_dir": case_dir,
        })
    finally:
        os.chdir(orig_dir)
        print(f"  >>> FINE CASO {idx+1}")

elapsed = (datetime.now() - START_TIME).total_seconds()
n_conv  = sum(r["converged"] for r in results_log)
print(f"\n{'─'*60}")
print(f"Completato in {elapsed:.0f}s")
print(f"Convergenti : {n_conv} / {len(results_log)}")
print(f"Non conv.   : {len(results_log) - n_conv} / {len(results_log)}")

Inizio analisi: 23:05:31
Casi totali   : 3

[0001/3] hover_ap0_bp0_V14_tFL90_tFR90_wFL4000_wFR4000_wRL4000_wRR4000_dep0_drp0_dap0
  >>> INIZIO CASO 1
  >>> DENTRO TRY
  >>> set_tilt FL
  >>> set_tilt FR
  >>> Update OK
  >>> set_rotor_params
  >>> rotori OK
  >>> CS OK
  >>> VSPAEROSweep start
/Users/primianodaddetta/anaconda3/envs/vsppytools/lib/python3.11/site-packages/openvsp/vspaero -omp 8 -stab /Users/primianodaddetta/Desktop/modello_vsp/provaprop
Setting Sref to: 0.980000 
Setting Cref to: 0.350000 
Setting Bref to: 2.800000 
Setting X_cg to: 0.484944 
Setting Y_cg to: 0.000038 
Setting Z_cg to: 0.842549 
Number of Machs: 1 
Mach: 0.041140 
Number of AoAs: 1 
AoA: 0.000000 
Number of Betas: 1 
Beta: 0.000000 
Number of ReCs: 1 
ReC: 500000.000000 
Setting Vinf to: 14.000000 
Setting Rho to: 1.225000 
Setting StallModel to: 0 
Setting Clo2D to: 0.000000 
Setting CLMax2D to: 1.000000 
Setting Symmetry to: 0 
Setting FreezeMultiPoleAtIteration to: 10000 
Setting FreezeWakeAtIteratio

ERROR 7: Could not open History file: /Users/primianodaddetta/Desktop/modello_vsp/provaprop.history
	File: /Users/runner/work/OpenVSP/OpenVSP/src/geom_core/VSPAEROMgr.cpp 	Line:2296
ERROR 7: Could not open Load file: /Users/primianodaddetta/Desktop/modello_vsp/provaprop.lod
	File: /Users/runner/work/OpenVSP/OpenVSP/src/geom_core/VSPAEROMgr.cpp 	Line:2883

Error: Aerothermal database (*.adb) file not found. Execute VSPAERO before running the CpSlicer


  >>> DOPO EXECANALYSIS rid=OPYPMZJ
  ExecAnalysis completato, rid=OPYPMZJ
  [WARN] hist non trovato in model_dir: /Users/primianodaddetta/Desktop/modello_vsp/provaprop.history
  stab copiato: /Users/primianodaddetta/Desktop/modello_vsp/hover/hover_ap0_bp0_V14_tFL90_tFR90_wFL4000_wFR4000_wRL4000_wRR4000_dep0_drp0_dap0/hover_ap0_bp0_V14_tFL90_tFR90_wFL4000_wFR4000_wRL4000_wRR4000_dep0_drp0_dap0.stab
no-conv | iter=0 | blocks=0 | α=+0.0° β=+0.0° V=14m/s
  >>> FINE CASO 1
[0002/3] transition_ap5_bp0_V16_tFL30_tFR30_wFL3000_wFR3000_wRL3000_wRR3000_dep5_drp0_dap
  >>> INIZIO CASO 2
  >>> DENTRO TRY
  >>> set_tilt FL
  >>> set_tilt FR
  >>> Update OK
  >>> set_rotor_params
  >>> rotori OK
  >>> CS OK
  >>> VSPAEROSweep start
/Users/primianodaddetta/anaconda3/envs/vsppytools/lib/python3.11/site-packages/openvsp/vspaero -omp 8 -stab /Users/primianodaddetta/Desktop/modello_vsp/provaprop
Setting Sref to: 0.980000 
Setting Cref to: 0.350000 
Setting Bref to: 2.800000 
Setting X_cg to: 0.484944 
S

ERROR 7: Could not open History file: /Users/primianodaddetta/Desktop/modello_vsp/provaprop.history
	File: /Users/runner/work/OpenVSP/OpenVSP/src/geom_core/VSPAEROMgr.cpp 	Line:2296
ERROR 7: Could not open Load file: /Users/primianodaddetta/Desktop/modello_vsp/provaprop.lod
	File: /Users/runner/work/OpenVSP/OpenVSP/src/geom_core/VSPAEROMgr.cpp 	Line:2883

Error: Aerothermal database (*.adb) file not found. Execute VSPAERO before running the CpSlicer


  >>> DOPO EXECANALYSIS rid=WFLBUGH
  ExecAnalysis completato, rid=WFLBUGH
  [WARN] hist non trovato in model_dir: /Users/primianodaddetta/Desktop/modello_vsp/provaprop.history
  stab copiato: /Users/primianodaddetta/Desktop/modello_vsp/transition/transition_ap5_bp0_V16_tFL30_tFR30_wFL3000_wFR3000_wRL3000_wRR3000_dep5_drp0_dap0/transition_ap5_bp0_V16_tFL30_tFR30_wFL3000_wFR3000_wRL3000_wRR3000_dep5_drp0_dap0.stab
no-conv | iter=0 | blocks=0 | α=+5.0° β=+0.0° V=16m/s
  >>> FINE CASO 2
[0003/3] cruise_ap0_bp0_V22_tFL0_tFR0_wFL2000_wFR2000_wRL1_wRR1_dep0_drp0_dap0
  >>> INIZIO CASO 3
  >>> DENTRO TRY
  >>> set_tilt FL
  >>> set_tilt FR
  >>> Update OK
  >>> set_rotor_params
  >>> rotori OK
  >>> CS OK
  >>> VSPAEROSweep start
/Users/primianodaddetta/anaconda3/envs/vsppytools/lib/python3.11/site-packages/openvsp/vspaero -omp 8 -stab /Users/primianodaddetta/Desktop/modello_vsp/provaprop
Setting Sref to: 0.980000 
Setting Cref to: 0.350000 
Setting Bref to: 2.800000 
Setting X_cg to: 0.48494

In [10]:
# fermare monitor
_monitor_active = False
print("Monitor fermato.")

Monitor fermato.


# DISACCOPPIAMENTO SPINTA DA RISULTANTE AERODINAMICA
OpenVSP restituisce in output le forze scomposte in assi vento e assi corpo indifferentemente dalla loro natura. Qui si prova a 'disaccoppiare' quelle dei corpi rotanti da quelle delle superficie portanti, per quanto possibile. E si procede alla costruzione del database.

In [11]:

import pandas as pd

#  POSIZIONI PROPELLER IN ASSI BODY [x, y, z] (m)
#  Aggiornare con le coordinate reali del tuo modello.
#  Origine = CG del velivolo.
#  Convenzione OpenVSP: x verso il muso, z verso l'alto.

# DA IMPORTARE CORRETTAMENTE DA MODELLO VSP !!!!!!!
PROP_POSITIONS = {
    "FL": (0.42, -0.70, 0.8330),
    "FR": (0.42,  0.70, 0.8330),
    "RL": ( 1, -0.50, 0.8330),
    "RR": ( 1,  0.50, 0.8330),
}

def compute_thrust_wrench(case):
    """
    Calcola forza e momento di spinta totale in assi body.
    
    Convenzione tilt:
      tilt=0°  → spinta lungo +Z body (hovering, disco orizzontale)
      tilt=90° → spinta lungo -X body (crociera, disco verticale)
    
    Restituisce dict con Fx,Fy,Fz [N] e Mx,My,Mz [N·m] da spinta pura.
    """
    Fx_tot = Fy_tot = Fz_tot = 0.0
    Mx_tot = My_tot = Mz_tot = 0.0

    tilt_rad = {
        "FL": np.radians(case["tilt_FL"]),
        "FR": np.radians(case["tilt_FR"]),
        "RL": 0.0,
        "RR": 0.0,
    }

    for key in ["FL", "FR", "RL", "RR"]:
        T     = case[f"T_{key}"]          # [N] dalla PropellerMap
        theta = tilt_rad[key]
        # Proiezione in body axes
        Fx = -T * np.cos(theta)  
        Fz =  T * np.sin(theta)   
        Fy =   0.0
        rx, ry, rz = PROP_POSITIONS[key]
        # Momento 
        Mx_tot += ry * Fz - rz * Fy
        My_tot += rz * Fx - rx * Fz
        Mz_tot += rx * Fy - ry * Fx
        Fx_tot += Fx
        Fz_tot += Fz

    return dict(Fx=Fx_tot, Fy=Fy_tot, Fz=Fz_tot,
                Mx=Mx_tot, My=My_tot, Mz=Mz_tot)


def subtract_thrust(coeff_block, case):
    """
    Sottrae il contributo di spinta pura ai coefficienti VSPAero.

    CF_aero = CF_vsp - F_thrust / (q_inf * Sref)
    CMy_aero = CMy_vsp - My_thrust / (q_inf * Sref * Cref)
    CMx/z_aero = CM_vsp - M / (q_inf * Sref * Bref)

    Il guadagno aerodinamico da slipstream/suction rimane
    accoppiato ai coefficienti aerodinamici (scelta fisica corretta).
    RPM=0 → nessuna sottrazione (propeller spento).
    """
    vinf = case["vinf"]
    if vinf < 0.5:   # hovering puro: evita divisione per zero
        aero = dict(coeff_block)
        for sfx in ["CFx","CFy","CFz","CMx","CMy","CMz","CL","CD"]:
            aero[f"{sfx}_aero"] = coeff_block.get(sfx, float("nan"))
        return aero

    q_inf  = 0.5 * RHO * vinf**2
    thrust = compute_thrust_wrench(case)

    aero = dict(coeff_block)
    # Forze adimensionalizzate con q*Sref
    aero["CFx_aero"] = coeff_block.get("CFx", 0.0) - thrust["Fx"] / (q_inf * SREF)
    aero["CFy_aero"] = coeff_block.get("CFy", 0.0) - thrust["Fy"] / (q_inf * SREF)
    aero["CFz_aero"] = coeff_block.get("CFz", 0.0) - thrust["Fz"] / (q_inf * SREF)
    # Momenti: CMy → Cref, CMx/CMz → Bref
    aero["CMx_aero"] = coeff_block.get("CMx", 0.0) - thrust["Mx"] / (q_inf * SREF * BREF)
    aero["CMy_aero"] = coeff_block.get("CMy", 0.0) - thrust["My"] / (q_inf * SREF * CREF)
    aero["CMz_aero"] = coeff_block.get("CMz", 0.0) - thrust["Mz"] / (q_inf * SREF * BREF)
    # CL e CD (assi vento — proiezione di CFz e CFx)
    aero["CL_aero"]  = coeff_block.get("CL", 0.0) - thrust["Fz"] / (q_inf * SREF)
    aero["CD_aero"]  = coeff_block.get("CD", 0.0) + thrust["Fx"] / (q_inf * SREF)
    return aero

# Disaccoppiamento derivate da spinta
def subtract_thrust_derivs_block(case, block):
    """
    Sottrae il contributo di spinta alle derivate rispetto ad alpha.
    Le derivate rispetto a beta, p, q, r lasciate invariate (prima approssimazione).
    Restituisce dict con chiavi *Alpha corrette + tutte le altre invariate.
    """
    vinf = case.get('vinf', 0.0)
    if vinf < 0.5:
        return {k: block.get(k, float('nan')) for k in block}

    alpha_rad = np.radians(case.get('alpha', 0.0))
    q_inf     = 0.5 * RHO * vinf**2
    thrust    = compute_thrust_wrench(case)
    Fx, Fz    = thrust['Fx'], thrust['Fz']

    # derivate di CL_thrust e CD_thrust rispetto ad alpha
    # CL_thrust = ( Fz*cos(a) - Fx*sin(a)) / (q*S)  →  dCL/da = (-Fz*sin(a) - Fx*cos(a))/(q*S)
    # CD_thrust = ( Fz*sin(a) + Fx*cos(a)) / (q*S)  →  dCD/da = ( Fz*cos(a) - Fx*sin(a))/(q*S)
    # CFz_thrust = Fz/(q*S) (body axis, non ruota) → dCFz/da = 0
    dCL_da  = (-Fz*np.sin(alpha_rad) - Fx*np.cos(alpha_rad)) / (q_inf * SREF) / np.radians(1)
    dCD_da  = ( Fz*np.cos(alpha_rad) - Fx*np.sin(alpha_rad)) / (q_inf * SREF) / np.radians(1)
    dCFx_da = ( Fz*np.sin(alpha_rad) - Fx*np.cos(alpha_rad)) / (q_inf * SREF) / np.radians(1)
    dCFz_da = 0.0   # body axis, spinta non ruota con alpha

    # momento di spinta: d(My_thrust)/da = 0 se posizione propulsori fissa
    # (la spinta in body axis non cambia con alpha, solo la proiezione in wind axis cambia)
    dCMy_da = (-(thrust['My'] / (q_inf * SREF * CREF)) * 0)  # = 0

    out = dict(block)   # copia tutti i valori originali
    out['CL_Alpha']  = block.get('CL_Alpha',  float('nan')) - dCL_da
    out['CD_Alpha']  = block.get('CD_Alpha',  float('nan')) - dCD_da
    out['CFx_Alpha'] = block.get('CFx_Alpha', float('nan')) - dCFx_da
    out['CFz_Alpha'] = block.get('CFz_Alpha', float('nan')) - dCFz_da
    # CMmAlpha, CMxAlpha, CMzAlpha, CSAlpha, etc. → nessuna correzione (prima approssimazione)
    return out


# Costruzione DataFrame database 
print("Costruzione database aerodinamico...")
db_rows = []

for log in results_log:
    if not log["converged"]:
        continue   # escludi non convergenti dal database

    cname  = log["name"]
    blocks = stab_results.get(cname, [])
    if not blocks:
        continue

    for block in blocks:
        # Parametri di volo dal log
        row = {
            "phase"   : log["phase"],
            "alpha"   : log["alpha"],
            "beta"    : log["beta"],
            "vinf"    : log["vinf"],
            "Mach"    : log["Mach"],
            "Re"      : log["Re"],
            "tilt_FL" : log["tilt_FL"],
            "tilt_FR" : log["tilt_FR"],
            "rpm_FL"  : log["rpm_FL"],
            "rpm_FR"  : log["rpm_FR"],
            "rpm_RL"  : log["rpm_RL"],
            "rpm_RR"  : log["rpm_RR"],
            "delta_e" : log["delta_e"],
            "delta_r" : log["delta_r"],
            "delta_a" : log["delta_a"],
            # alpha effettivo letto dal .stab (potrebbe differire leggermente)
            "alpha_stab" : block.get("AoA", log["alpha"]),
            # spinta per propeller (assoluta)
            "T_FL"  : log["T_FL"],  "T_FR"  : log["T_FR"],
            "T_RL"  : log["T_RL"],  "T_RR"  : log["T_RR"],
            "Q_FL"  : log["Q_FL"],  "Q_FR"  : log["Q_FR"],
            "Q_RL"  : log["Q_RL"],  "Q_RR"  : log["Q_RR"],
            "CT_FL" : log["ct_FL"], "CT_FR" : log["ct_FR"],
            "CT_RL" : log["ct_RL"], "CT_RR" : log["ct_RR"],
            "CP_FL" : log["cp_FL"], "CP_FR" : log["cp_FR"],
            "CP_RL" : log["cp_RL"], "CP_RR" : log["cp_RR"],
        }

        # Coefficienti e derivate dal .stab 
        skip = {"Sref","Cref","Bref","Xcg","Ycg","Zcg",
                "Mach_cond","AoA","Beta_cond","Rho","Vinf_cond"}
        for k, v in block.items():
            if k not in skip:
                row[k] = v

        # Separazione contributo spinta
        aero = subtract_thrust(block, log)
        for k in ["CFx_aero","CFy_aero","CFz_aero",
                  "CMx_aero","CMy_aero","CMz_aero",
                  "CL_aero","CD_aero"]:
            row[k] = aero.get(k, float("nan"))

        aero_derivs = subtract_thrust_derivs_block(log, block)
        for deriv_key in ('CL_Alpha','CD_Alpha','CFx_Alpha','CFz_Alpha'):
            row[deriv_key + 'aero'] = aero_derivs.get(deriv_key, float('nan'))
        # Le altre derivate (Beta, p, q, r) vengono già lette da block direttamente
        # ma puoi aggiungere alias "aero" anche per le altre se vuoi uniformità:
        for deriv_key in ('CS_Beta','CMl_Beta','CMn_Beta','CMm_Alpha',
                        'CMp','CMr','CMq','C_q'):
            row[deriv_key + '_aero'] = block.get(deriv_key, float('nan'))

        db_rows.append(row)

df_db = pd.DataFrame(db_rows)

print(f"\nDatabase costruito:")
print(f"  Righe (casi convergenti) : {len(df_db)}")
print(f"  Colonne                  : {len(df_db.columns)}")
print(f"\nColonne presenti:")
for i, col in enumerate(df_db.columns):
    print(f"  {i+1:3d}. {col}")


Costruzione database aerodinamico...

Database costruito:
  Righe (casi convergenti) : 1
  Colonne                  : 150

Colonne presenti:
    1. phase
    2. alpha
    3. beta
    4. vinf
    5. Mach
    6. Re
    7. tilt_FL
    8. tilt_FR
    9. rpm_FL
   10. rpm_FR
   11. rpm_RL
   12. rpm_RR
   13. delta_e
   14. delta_r
   15. delta_a
   16. alpha_stab
   17. T_FL
   18. T_FR
   19. T_RL
   20. T_RR
   21. Q_FL
   22. Q_FR
   23. Q_RL
   24. Q_RR
   25. CT_FL
   26. CT_FR
   27. CT_RL
   28. CT_RR
   29. CP_FL
   30. CP_FR
   31. CP_RL
   32. CP_RR
   33. SM
   34. X_np
   35. CFx
   36. CFx_Alpha
   37. CFx_Beta
   38. CFx_p
   39. CFx_q
   40. CFx_r
   41. CFx_Mach
   42. CFx_U
   43. CFy
   44. CFy_Alpha
   45. CFy_Beta
   46. CFy_p
   47. CFy_q
   48. CFy_r
   49. CFy_Mach
   50. CFy_U
   51. CFz
   52. CFz_Alpha
   53. CFz_Beta
   54. CFz_p
   55. CFz_q
   56. CFz_r
   57. CFz_Mach
   58. CFz_U
   59. CMx
   60. CMx_Alpha
   61. CMx_Beta
   62. CMx_p
   63. CMx_q
   64. CMx

# DERIVATE DI STABILITA'
Estrazione da database costruito nella cella precedente

In [ ]:

stab_files = glob.glob(os.path.join(OUTPUT_DIR, "**/*.stab"), recursive=True)
print(stab_files)

DERIVATIVES = ["CL", "CD", "CS",
               "CL_Alpha", "CD_Alpha", "CMm_Alpha",        # derivate rispetto alpha
               "CS_Beta", "CMl_Beta", "CMn_Beta",          # derivate rispetto beta
               "CMl_p", "CMn_p",                           # derivate rispetto p
               "CMm_q", "CL_q",                            # derivate rispetto q
               "CMl_r", "CMn_r",                           # derivate rispetto r
               "SM", "X_np"]


print(f"\n{'Derivata':<10}", end="")
for case in ALL_CASES:
    print(f"{case['name']:>15}", end="")
print()
print("-" * (10 + 15 * len(ALL_CASES)))

for deriv in DERIVATIVES:
    print(f"{deriv:<10}", end="")
    for case in ALL_CASES:
        block = stab_results.get(case["name"], [{}])
        if block:
            aero_derivs = subtract_thrust_derivs_block(case, block[0])
            val = aero_derivs.get(deriv, float('nan'))
        else:
            val = float('nan')
        print(f"{val:>15.5f}", end="")
    print()

print(stab_results)


['/Users/primianodaddetta/Desktop/modello_vsp/prova.stab', '/Users/primianodaddetta/Desktop/modello_vsp/provaprop 2.stab', '/Users/primianodaddetta/Desktop/modello_vsp/prova.aerocenter.stab', '/Users/primianodaddetta/Desktop/modello_vsp/provaprop 3.stab', '/Users/primianodaddetta/Desktop/modello_vsp/transition/transition_ap5_bp0_V16_tFL30_tFR30_wFL3500_wFR3500_wRL3500_wRR3500_dep5_drp0_dap0/transition_ap5_bp0_V16_tFL30_tFR30_wFL3500_wFR3500_wRL3500_wRR3500_dep5_drp0_dap0.stab', '/Users/primianodaddetta/Desktop/modello_vsp/transition/transition_ap5_bp0_V16_tFL30_tFR30_wFL3000_wFR3000_wRL3000_wRR3000_dep5_drp0_dap0/transition_ap5_bp0_V16_tFL30_tFR30_wFL3000_wFR3000_wRL3000_wRR3000_dep5_drp0_dap0.stab', '/Users/primianodaddetta/Desktop/modello_vsp/hover/hover.stab', '/Users/primianodaddetta/Desktop/modello_vsp/hover/hover_ap0_bp0_V14_tFL90_tFR90_wFL3000_wFR3000_wRL3000_wRR3000_dep0_drp0_dap0/hover_ap0_bp0_V14_tFL90_tFR90_wFL3000_wFR3000_wRL3000_wRR3000_dep0_drp0_dap0.stab', '/Users/primia

# EXPORT CSV
Export, sicuramente migliorabile.

In [13]:

import json
from datetime import datetime

#  EXPORT DATABASE AERODINAMICO

# CSV principale — solo casi convergenti 
csv_path = os.path.join(OUTPUT_DIR, "aero_database.csv")
df_db.to_csv(csv_path, index=False)
print(f" Database CSV        : {csv_path}")
print(f"  {len(df_db)} righe  |  {len(df_db.columns)} colonne")

# CSV run log — tutti i casi (convergenti + non) 
df_log = pd.DataFrame(results_log)
log_path = os.path.join(OUTPUT_DIR, "run_log.csv")
df_log.to_csv(log_path, index=False)
print(f" Run log CSV         : {log_path}")

n_conv    = int(df_log["converged"].sum())
n_total   = len(df_log)
n_hover   = int((df_log["phase"] == "hover").sum())
n_trans   = int((df_log["phase"] == "transition").sum())
n_cruise  = int((df_log["phase"] == "cruise").sum())
print(f"  Convergenti: {n_conv}/{n_total}  "
      f"(hover={n_hover}, transizione={n_trans}, crociera={n_cruise})")

# JSON metadata 
meta = {
    "generated"      : datetime.now().isoformat(),
    "model_path"     : MODEL_PATH,
    "prop_mode"      : propmode,
    "total_cases"    : n_total,
    "converged"      : n_conv,
    "db_rows"        : len(df_db),
    "db_columns"     : list(df_db.columns),
    "phases"         : list(df_db["phase"].unique()) if len(df_db) > 0 else [],
    "reference"      : {"Sref": SREF, "Cref": CREF, "Bref": BREF},
    "atmosphere"     : {"rho": RHO, "mu": MU, "a_sound": A_SOUND},
    "prop_positions" : PROP_POSITIONS,
    "convergence"    : {"keys": CONV_KEYS,"min_iter": MIN_ITER},
    # Statistiche per fase
    "stats_by_phase" : {
        phase: {
            "total"    : int((df_log["phase"] == phase).sum()),
            "converged": int(df_log[df_log["phase"] == phase]["converged"].sum()),
        }
        for phase in ["hover", "transition", "cruise"]
    },
}
meta_path = os.path.join(OUTPUT_DIR, "aero_database_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"✓ Metadata JSON       : {meta_path}")

# Anteprima database 
base_cols = ["phase", "alpha", "beta", "vinf",
             "tilt_FL", "tilt_FR",
             "rpm_FL", "rpm_FR", "rpm_RL", "rpm_RR",
             "delta_e", "delta_r", "delta_a",
             "CL", "CD", "CS", "CMl", "CMm", "CMn",
             "CL_aero", "CD_aero",
             "T_FL", "T_FR", "T_RL", "T_RR"]
avail = [c for c in base_cols if c in df_db.columns]

print(f"\nAnteprima database ({len(df_db)} righe):")
print(df_db[avail].head(10).to_string(index=False))

# Riepilogo non convergenti (per diagnostica) 
df_noconv = df_log[~df_log["converged"]]
if len(df_noconv) > 0:
    noconv_path = os.path.join(OUTPUT_DIR, "non_convergenti.csv")
    df_noconv.to_csv(noconv_path, index=False)
    print(f"\n Casi non convergenti ({len(df_noconv)}) salvati in:")
    print(f"   {noconv_path}")
    print(df_noconv[["name","phase","alpha","beta","vinf",
                      "conv_reason","n_iter"]].to_string(index=False))
else:
    print("\n✓ Tutti i casi sono convergenti.")


 Database CSV        : /Users/primianodaddetta/Desktop/modello_vsp/aero_database.csv
  1 righe  |  150 colonne
 Run log CSV         : /Users/primianodaddetta/Desktop/modello_vsp/run_log.csv
  Convergenti: 1/3  (hover=1, transizione=1, crociera=1)
✓ Metadata JSON       : /Users/primianodaddetta/Desktop/modello_vsp/aero_database_meta.json

Anteprima database (1 righe):
 phase  alpha  beta  vinf  tilt_FL  tilt_FR  rpm_FL  rpm_FR  rpm_RL  rpm_RR  delta_e  delta_r  delta_a       CL       CD      CS       CMl       CMm      CMn  CL_aero  CD_aero     T_FL     T_FR         T_RL         T_RR
cruise    0.0   0.0  22.0      0.0      0.0  2000.0  2000.0     1.0     1.0      0.0      0.0      0.0 0.474321 0.038336 0.01175 -0.004699 -0.042689 0.002494 0.474321 0.024383 2.026752 2.026752 5.019813e-07 5.019813e-07

 Casi non convergenti (2) salvati in:
   /Users/primianodaddetta/Desktop/modello_vsp/non_convergenti.csv
                                                                             name   